In [2]:
from pathlib import Path

WIKI_DIR = Path("../wiki/pages")   # 팀 위키로 바꿀 때는 Path("../../team/wiki")
print("현재 위치:", Path.cwd())
print("폴더 존재:", WIKI_DIR.exists())
print(sorted(p.name for p in WIKI_DIR.glob("**/*.md")))

현재 위치: c:\Users\bagaj\Documents\GitHub\DockerPractice\Study-service_basic\members\huiji\week3
폴더 존재: True
['image-vs-container.md', 'rag-pipeline.md', 'when-to-use-async.md']


In [4]:
%pip install langchain_community

  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached langchain_classic-1.0.8-py3-none-any.whl.metadata (5.1 kB)
  Using cached langchain_core-1.7.4-py3-none-any.whl.metadata (4.8 kB)
  Using cached langsmith-0.14.17-py3-none-any.whl.metadata (39 kB)
  Using cached pydantic_settings-2.16.2-py3-none-any.whl.metadata (4.0 kB)
  Using cached pyyaml-6.0.3-cp314-cp314-win_amd64.whl.metadata (2.4 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached tenacity-9.2.1-py3-none-any.whl.metadata (1.4 kB)
  Using cached aiohappyeyeballs-2.7.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached langchain_text_splitters-1.1.3-py3-none-any.whl.metadata (3.6 kB)
  Using cached pydantic-2.14.2-py3-none-any.whl.metadata (137 kB)
  Using cached cert

In [20]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(
    str(WIKI_DIR),
    glob="**/*.md",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},   # 한글 깨짐 방지
)
docs = loader.load()
print(len(docs), "개 문서를 읽었어요")

6 개 문서를 읽었어요


In [21]:
d = docs[0]
print("타입:", type(d).__name__)
print("metadata:", d.metadata)
print("본문 길이:", len(d.page_content), "자")
print("--- 본문 앞부분 ---")
print(d.page_content[:200])

타입: Document
metadata: {'source': '..\\wiki\\pages\\chunking.md'}
본문 길이: 161 자
--- 본문 앞부분 ---
# 청킹 (Chunking)

## 한 줄 정의
긴 문서를 chunk_size 단위로 잘라 임베딩과 검색의 단위로 만드는 것

## 왜 필요한가
통째로 하는 임베딩은 검색이 뭉툭해지고, 너무 잘게 자르면 문맥이 끊긴다

## 예제
rag.ipynb 참고

## 내가 막힌 점

## 출처



In [22]:
for d in docs:
    d.metadata["page"] = Path(d.metadata["source"]).stem   # 확장자 뺀 파일 이름

for d in docs:
    print(d.metadata["page"], "|", len(d.page_content), "자")

chunking | 161 자
document-loading | 166 자
embedding-vectordb | 175 자
image-vs-container | 425 자
rag-pipeline | 268 자
when-to-use-async | 455 자


In [8]:
%pip install langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [23]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)

print(len(docs), "개 문서 →", len(chunks), "개 청크")

6 개 문서 → 6 개 청크


In [24]:
print(chunks[0].metadata)
print(len(chunks[0].page_content), "자")
print(chunks[0].page_content[:150])

{'source': '..\\wiki\\pages\\chunking.md', 'page': 'chunking'}
160 자
# 청킹 (Chunking)

## 한 줄 정의
긴 문서를 chunk_size 단위로 잘라 임베딩과 검색의 단위로 만드는 것

## 왜 필요한가
통째로 하는 임베딩은 검색이 뭉툭해지고, 너무 잘게 자르면 문맥이 끊긴다

## 예제
rag.ipynb 참고

## 내가 막


In [25]:
def add_chunk_ids(chunks):
    counter = {}
    for c in chunks:
        page = c.metadata["page"]
        counter[page] = counter.get(page, 0) + 1
        c.metadata["id"] = f"{page}::{counter[page]:03d}"
    return chunks

chunks = add_chunk_ids(chunks)

for c in chunks[:5]:
    print(c.metadata["id"], "|", c.page_content[:40].replace("\n", " "))

chunking::001 | # 청킹 (Chunking)  ## 한 줄 정의 긴 문서를 chunk_s
document-loading::001 | # 문서 로드 (Document Loader)  ## 한 줄 정의 파일을
embedding-vectordb::001 | # 임베딩과 벡터 DB  ## 한 줄 정의 글을 벡터로 바꾸고, 질문 벡
image-vs-container::001 | # 이미지와 컨테이너의 차이 이미지는 컨테이너를 실행하기 위해 필요한 파
rag-pipeline::001 | # 정의 LLM이 답하기 전에 질문과 관련된 문서를 먼저 검색하고 그 내


In [26]:
ids = [c.metadata["id"] for c in chunks]

assert len(ids) == len(set(ids)), "중복 ID가 있어요!"
assert all("::" in i for i in ids), "형식이 다른 ID가 있어요!"
print("ID", len(ids), "개 모두 고유해요")

ID 6 개 모두 고유해요


In [13]:
for size, overlap in [(200, 20), (500, 50), (1000, 100)]:
    s = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=overlap)
    n = len(s.split_documents(docs))
    print(f"chunk_size={size:>4}, overlap={overlap:>3} → {n}개 청크")

chunk_size= 200, overlap= 20 → 10개 청크
chunk_size= 500, overlap= 50 → 3개 청크
chunk_size=1000, overlap=100 → 3개 청크


In [14]:
%pip install langchain-chroma langchain-huggingface sentence-transformers

  Using cached langchain_chroma-1.1.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached langchain_huggingface-1.2.3-py3-none-any.whl.metadata (4.3 kB)
  Using cached sentence_transformers-6.2.0-py3-none-any.whl.metadata (20 kB)
  Using cached chromadb-1.5.9-cp39-abi3-win_amd64.whl.metadata (5.1 kB)
  Using cached build-1.6.1-py3-none-any.whl.metadata (5.6 kB)
  Using cached uvicorn-0.54.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached opentelemetry_api-1.45.2-py3-none-any.whl.metadata (1.4 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.45.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached opentelemetry_sdk-1.45.2-py3-none-any.whl.metadata (1.6 kB)
  Using cached tokenizers-0.23.3-cp310-abi3-win_amd64.whl.metadata (10 kB)
  Using cached pypika-0.51.1-py2.py3-none-any.whl.metadata (51 kB)
  Using cached tqdm-4.70.1-py3-none-any.whl.metadata (57 kB)
  Using cached overrides-7.7.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached importlib_resources-7.1.0-py3-none-any.whl.

In [15]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    encode_kwargs={"normalize_embeddings": True},
)
print("모델 로드 완료")

c:\Users\bagaj\Documents\GitHub\DockerPractice\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\bagaj\Documents\GitHub\DockerPractice\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\bagaj\.cache\huggingface\hub\models--BAAI--bge-m3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. I

모델 로드 완료


In [27]:
import numpy as np

def sim(a, b):
    va, vb = embeddings.embed_query(a), embeddings.embed_query(b)
    return float(np.dot(va, vb))   # 정규화했으니 내적 = 유사도 (1에 가까울수록 비슷)

print("벡터 길이:", len(embeddings.embed_query("테스트")))
print("컨테이너 vs 도커 실행 단위:", round(sim("컨테이너가 뭐야?", "도커의 실행 단위는?"), 3))
print("컨테이너 vs 쿠버네티스 Service:", round(sim("컨테이너가 뭐야?", "쿠버네티스 Service란?"), 3))

벡터 길이: 1024
컨테이너 vs 도커 실행 단위: 0.396
컨테이너 vs 쿠버네티스 Service: 0.415


In [28]:
from langchain_chroma import Chroma

def build_store(chunks, name="wiki_500_50"):
    return Chroma.from_documents(
        chunks,
        embeddings,
        ids=[c.metadata["id"] for c in chunks],    # 3단계에서 만든 ID를 그대로 사용
        collection_name=name,
        persist_directory=f"./chroma_db/{name}",   # 디스크에 저장
    )

vs = build_store(chunks)
print(vs._collection.count(), "개 저장됨 / 청크", len(chunks), "개")

6 개 저장됨 / 청크 6 개


In [29]:
got = vs.get(limit=3, include=["documents", "metadatas"])
for i, m, d in zip(got["ids"], got["metadatas"], got["documents"]):
    print(i, "|", m.get("page"), "|", d[:40].replace("\n", " "))

image-vs-container::001 | image-vs-container | # 이미지와 컨테이너의 차이 이미지는 컨테이너를 실행하기 위해 필요한 파
rag-pipeline::001 | rag-pipeline | # 정의 LLM이 답하기 전에 질문과 관련된 문서를 먼저 검색하고 그 내
when-to-use-async::001 | when-to-use-async | # async 함수는 언제 필요한가  ## 한 줄 정의 async def


In [30]:
vs2 = Chroma(
    collection_name="wiki_500_50",
    embedding_function=embeddings,
    persist_directory="./chroma_db/wiki_500_50",
)
print(vs2._collection.count(), "개 (재로드 후)")

6 개 (재로드 후)


In [31]:
retriever = vs.as_retriever(search_kwargs={"k": 5})

results = retriever.invoke("Docker 이미지와 컨테이너의 차이는?")
for rank, d in enumerate(results, 1):
    print(rank, d.metadata["id"], "|", d.page_content[:50].replace("\n", " "))

1 image-vs-container::001 | # 이미지와 컨테이너의 차이 이미지는 컨테이너를 실행하기 위해 필요한 파일, 라이브러리, 
2 embedding-vectordb::001 | # 임베딩과 벡터 DB  ## 한 줄 정의 글을 벡터로 바꾸고, 질문 벡터와 가까운 벡터를
3 document-loading::001 | # 문서 로드 (Document Loader)  ## 한 줄 정의 파일을 읽어 page_c
4 chunking::001 | # 청킹 (Chunking)  ## 한 줄 정의 긴 문서를 chunk_size 단위로 잘라
5 when-to-use-async::001 | # async 함수는 언제 필요한가  ## 한 줄 정의 async def는 await로 기


In [32]:
def show(question, k=5):
    print("질문:", question)
    for rank, (d, dist) in enumerate(vs.similarity_search_with_score(question, k=k), 1):
        print(f"  {rank}. 거리 {dist:.3f} | {d.metadata['id']} | {d.page_content[:40]!r}")
    print()

show("Docker 이미지와 컨테이너의 차이는?")

질문: Docker 이미지와 컨테이너의 차이는?
  1. 거리 0.446 | image-vs-container::001 | '# 이미지와 컨테이너의 차이\n이미지는 컨테이너를 실행하기 위해 필요한 파'
  2. 거리 1.155 | embedding-vectordb::001 | '# 임베딩과 벡터 DB\n\n## 한 줄 정의\n글을 벡터로 바꾸고, 질문 벡'
  3. 거리 1.242 | document-loading::001 | '# 문서 로드 (Document Loader)\n\n## 한 줄 정의\n파일을'
  4. 거리 1.272 | chunking::001 | '# 청킹 (Chunking)\n\n## 한 줄 정의\n긴 문서를 chunk_s'
  5. 거리 1.316 | when-to-use-async::001 | '# async 함수는 언제 필요한가\n\n## 한 줄 정의\nasync def'



In [33]:
show("컨테이너가 뭐야?")                # 쉬운 질문: 위키에 정확히 있는 내용
show("도커의 실행 단위는?")              # 표현을 바꾼 질문 (1-3에서 단어 겹침이 틀렸던 유형)
show("오늘 점심 뭐 먹지?")               # 위키에 없는 질문

질문: 컨테이너가 뭐야?
  1. 거리 0.795 | image-vs-container::001 | '# 이미지와 컨테이너의 차이\n이미지는 컨테이너를 실행하기 위해 필요한 파'
  2. 거리 1.139 | embedding-vectordb::001 | '# 임베딩과 벡터 DB\n\n## 한 줄 정의\n글을 벡터로 바꾸고, 질문 벡'
  3. 거리 1.213 | chunking::001 | '# 청킹 (Chunking)\n\n## 한 줄 정의\n긴 문서를 chunk_s'
  4. 거리 1.285 | document-loading::001 | '# 문서 로드 (Document Loader)\n\n## 한 줄 정의\n파일을'
  5. 거리 1.328 | rag-pipeline::001 | '# 정의\nLLM이 답하기 전에 질문과 관련된 문서를 먼저 검색하고 그 내'

질문: 도커의 실행 단위는?
  1. 거리 0.956 | image-vs-container::001 | '# 이미지와 컨테이너의 차이\n이미지는 컨테이너를 실행하기 위해 필요한 파'
  2. 거리 1.275 | chunking::001 | '# 청킹 (Chunking)\n\n## 한 줄 정의\n긴 문서를 chunk_s'
  3. 거리 1.288 | document-loading::001 | '# 문서 로드 (Document Loader)\n\n## 한 줄 정의\n파일을'
  4. 거리 1.337 | when-to-use-async::001 | '# async 함수는 언제 필요한가\n\n## 한 줄 정의\nasync def'
  5. 거리 1.362 | embedding-vectordb::001 | '# 임베딩과 벡터 DB\n\n## 한 줄 정의\n글을 벡터로 바꾸고, 질문 벡'

질문: 오늘 점심 뭐 먹지?
  1. 거리 1.292 | when-to-use-async::001 | '# async 함수는 언제 필요한가\n\n## 한 줄 정의\nasync def'
  2. 거리 1.307 | chunki

In [34]:
mini = [
    {"question": "컨테이너가 뭐야?",          "answer_page": "image-vs-container"},
    {"question": "RAG에서 청킹은 왜 해?",     "answer_page": "chunking"},
    {"question": "임베딩은 뭘 하는 거야?",    "answer_page": "embedding-vectordb"},
]

hits = 0
for q in mini:
    pages = [d.metadata["page"] for d in retriever.invoke(q["question"])]
    ok = q["answer_page"] in pages
    hits += ok
    print("O" if ok else "X", q["question"], "→", pages)

print(f"맞힌 비율: {hits}/{len(mini)}")

O 컨테이너가 뭐야? → ['image-vs-container', 'embedding-vectordb', 'chunking', 'document-loading', 'rag-pipeline']
O RAG에서 청킹은 왜 해? → ['chunking', 'rag-pipeline', 'when-to-use-async', 'embedding-vectordb', 'image-vs-container']
O 임베딩은 뭘 하는 거야? → ['chunking', 'embedding-vectordb', 'rag-pipeline', 'image-vs-container', 'when-to-use-async']
맞힌 비율: 3/3


In [35]:
for k in (1, 3, 5, 10):
    print(f"k={k}:", [d.metadata["id"] for d in vs.as_retriever(search_kwargs={"k": k}).invoke("컨테이너가 뭐야?")])

k=1: ['image-vs-container::001']
k=3: ['image-vs-container::001', 'embedding-vectordb::001', 'chunking::001']
k=5: ['image-vs-container::001', 'embedding-vectordb::001', 'chunking::001', 'document-loading::001', 'rag-pipeline::001']
k=10: ['image-vs-container::001', 'embedding-vectordb::001', 'chunking::001', 'document-loading::001', 'rag-pipeline::001', 'when-to-use-async::001']


In [37]:
%pip install langchain_google_genai

  Using cached filetype-1.2.0-py2.py3-none-any.whl.metadata (6.5 kB)
  Using cached google_genai-2.34.0-py3-none-any.whl.metadata (59 kB)
  Using cached google_auth-2.62.1-py3-none-any.whl.metadata (6.0 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached cryptography-50.0.2-cp311-abi3-win_amd64.whl.metadata (4.4 kB)
  Using cached pycparser-3.12-py3-none-any.whl.metadata (9.7 kB)
  Using cached pyasn1-0.6.5-py3-none-any.whl.metadata (8.4 kB)
   ---------------------------------------- 0.0/580.5 kB ? eta -:--:--
   ---------------------------------------- 580.5/580.5 kB 13.2 MB/s  0:00:00
Using cached filetype-1.2.0-py2.py3-none-any.whl (19 kB)
Using cached google_genai-2.34.0-py3-none-any.whl (1.3 MB)
Using cached google_auth-2.62.1-py3-none-any.whl (267 kB)
Using cached tenacity-9.1.4-py3-none-any.whl (28 kB)
Using cached cryptography-50.0.2-cp311-abi3-win_amd64.whl (3.8 MB)
Using ca

In [43]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()   # .env의 GOOGLE_API_KEY를 읽어요

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", temperature=0)
print(llm.invoke("한 문장으로 자기소개 해줘").content)

c:\Users\bagaj\Documents\GitHub\DockerPractice\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': '끊임없이 배우고 성장하며, 도움이 필요한 곳에 유용한 지식을 전하는 다재다능한 AI 어시스턴트입니다.', 'extras': {'signature': 'EmAKXgFpFH0TguwvfT6HtEfcNGJAk19FZ/iBegt5yecOfSwoHeDi0GJCZtQG+BJ8kuz12FCpeROFXpvU9Gj+pSzL3tm8wui5mUQBjPOBqCEZ1EtjFPTvHhrtMhAaVWL/qrY='}}]


In [44]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    "아래 문맥만 근거로 한국어로 답하세요.\n"
    "문맥에 답이 없으면 '문서에서 찾을 수 없어요'라고만 답하세요.\n\n"
    "문맥:\n{context}\n\n"
    "질문: {question}"
)

print(prompt.invoke({"context": "[테스트] 도커는 컨테이너 도구예요.", "question": "도커가 뭐야?"}).to_string())

Human: 아래 문맥만 근거로 한국어로 답하세요.
문맥에 답이 없으면 '문서에서 찾을 수 없어요'라고만 답하세요.

문맥:
[테스트] 도커는 컨테이너 도구예요.

질문: 도커가 뭐야?


In [45]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(f"[{d.metadata['page']}] {d.page_content}" for d in docs)

simple_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

print(simple_chain.invoke("Docker 이미지와 컨테이너의 차이는?"))

c:\Users\bagaj\Documents\GitHub\DockerPractice\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


이미지는 컨테이너를 실행하기 위해 필요한 파일, 라이브러리, 프로그램, 설정 등을 묶어 놓은 읽기 전용 템플릿이고, 컨테이너는 이미지를 기반으로 생성되는 격리된 실행 환경이다. (한 줄 정의: 이미지는 설계 도면이고 컨테이너는 도면대로 만든 것이다.)


In [46]:
from langchain_core.runnables import RunnableParallel

answer_chain = (
    (lambda x: {"context": format_docs(x["docs"]), "question": x["question"]})
    | prompt | llm | StrOutputParser()
)

rag = (
    RunnableParallel(docs=retriever, question=RunnablePassthrough())
    .assign(answer=answer_chain)
)

def ask(question):
    out = rag.invoke(question)
    sources = sorted({d.metadata["page"] for d in out["docs"]})
    return out["answer"], sources

In [47]:
for q in [
    "Docker 이미지와 컨테이너의 차이는?",   # 위키에 있는 질문
    "도커의 실행 단위는?",                   # 표현을 바꾼 질문
    "오늘 점심 뭐 먹지?",                    # 위키에 없는 질문
]:
    answer, sources = ask(q)
    print("Q:", q)
    print("A:", answer)
    print("출처:", sources, "\n")

c:\Users\bagaj\Documents\GitHub\DockerPractice\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Q: Docker 이미지와 컨테이너의 차이는?
A: 이미지는 컨테이너를 실행하기 위해 필요한 파일, 라이브러리, 프로그램, 설정 등을 묶어 놓은 읽기 전용 템플릿이고, 컨테이너는 이미지를 기반으로 생성되는 격리된 실행 환경이다. (한 줄 정의: 이미지는 설계 도면이고 컨테이너는 도면대로 만든 것이다.)
출처: ['chunking', 'document-loading', 'embedding-vectordb', 'image-vs-container', 'when-to-use-async'] 



c:\Users\bagaj\Documents\GitHub\DockerPractice\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Q: 도커의 실행 단위는?
A: 문서에서 찾을 수 없어요
출처: ['chunking', 'document-loading', 'embedding-vectordb', 'image-vs-container', 'when-to-use-async'] 



c:\Users\bagaj\Documents\GitHub\DockerPractice\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Q: 오늘 점심 뭐 먹지?
A: 문서에서 찾을 수 없어요
출처: ['chunking', 'document-loading', 'embedding-vectordb', 'rag-pipeline', 'when-to-use-async'] 



In [48]:
q = "Docker 이미지와 컨테이너의 차이는?"
docs_found = retriever.invoke(q)
print(prompt.invoke({"context": format_docs(docs_found), "question": q}).to_string())

Human: 아래 문맥만 근거로 한국어로 답하세요.
문맥에 답이 없으면 '문서에서 찾을 수 없어요'라고만 답하세요.

문맥:
[image-vs-container] # 이미지와 컨테이너의 차이
이미지는 컨테이너를 실행하기 위해 필요한 파일, 라이브러리, 프로그램, 설정 등을 묶어 놓은 읽기 전용 템플릿이고 컨테이너는 이미지를 기반으로 생성되는 격리된 실행 환경이다. 
## 한 줄 정의
이미지는 설계 도면이고 컨테이너는 도면대로 만든 것이다.
## 왜 필요한가
도커를 사용하는 이유는 서로 다른 운영체제나 실행 환경에서도 같은 설정으로 프로그램을 실행할 수 있도록 하기 위해서다.
## 이미지 크기 비교
| 이미지 | 베이스 | 크기(DISK USAGE) |
|---|---|---|
| app-full | python:3.11 | 1.64GB |
| app-slim | python:3.11-slim | 227MB |

## 내가 막힌 점

## 출처
- https://docs.docker.com/get-started/

[embedding-vectordb] # 임베딩과 벡터 DB

## 한 줄 정의
글을 벡터로 바꾸고, 질문 벡터와 가까운 벡터를 찾아 주는 저장소

## 왜 필요한가
단어가 달라도 뜻이 비슷한 문서를 찾기 위해

## 예제
rag.ipynb 참고

## 내가 막힌 점

## 출처
- BAAI/bge-m3 모델
- LangChain 문서: Chroma

[document-loading] # 문서 로드 (Document Loader)

## 한 줄 정의
파일을 읽어 page_content와 metadata를 가진 Document로 바꾸는 단계

## 왜 필요한가
metadata가 출처 표시와 평가에 쓰이기 때문

## 예제
rag.ipynb 참고

## 내가 막힌 점

## 출처

[chunking] # 청킹 (Chunking)

## 한 줄 정의
긴 문서를 chunk_size 단위로 잘라 임베딩과 검색의 단위로 만드는 것

## 왜 필요한가
통째로 하는

In [49]:
import json, time

def load_questions(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def evaluate(retriever, questions, k=5):
    hits, rr, times, misses = 0, 0.0, [], []
    for q in questions:
        t = time.perf_counter()
        docs = retriever.invoke(q["question"])[:k]
        times.append(time.perf_counter() - t)

        pages = [d.metadata["page"] for d in docs]
        if q["answer_page"] in pages:
            hits += 1
            rr += 1 / (pages.index(q["answer_page"]) + 1)   # 1위=1, 2위=0.5 ...
        else:
            misses.append((q["question"], q["answer_page"], pages))

    n = len(questions)
    return {
        "Recall@k": hits / n,
        "MRR@k": rr / n,
        "평균 응답(초)": sum(times) / n,
    }, misses

In [53]:
questions = load_questions("questions.jsonl")   # 연습용은 내 폴더, 팀 것은 ../../team/eval/questions.jsonl
retriever = vs.as_retriever(search_kwargs={"k": 5})

retriever.invoke("워밍업")   # 첫 호출은 느려서 시간 측정에서 빼요

score, misses = evaluate(retriever, questions, k=5)
for name, v in score.items():
    print(f"{name}: {v:.3f}")

print("\n--- 틀린 질문 ---")
for q, ans, got in misses:
    print("Q:", q, "| 정답:", ans, "| 검색됨:", got)

Recall@k: 1.000
MRR@k: 1.000
평균 응답(초): 0.157

--- 틀린 질문 ---


In [54]:
def build_retriever(chunk_size, chunk_overlap, k=5):
    s = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    ch = add_chunk_ids(s.split_documents(docs))
    store = build_store(ch, name=f"wiki_{chunk_size}_{chunk_overlap}")
    return store.as_retriever(search_kwargs={"k": k}), len(ch)

r200, n200 = build_retriever(200, 20)
r200.invoke("워밍업")
score200, _ = evaluate(r200, questions)
print(f"200/20 → 청크 {n200}개", {k: round(v, 3) for k, v in score200.items()})

200/20 → 청크 13개 {'Recall@k': 1.0, 'MRR@k': 0.833, '평균 응답(초)': 0.169}
